# EDA Framework

For future datasets, remember this sequence:

**Understand → Validate → Clean → Describe → Visualize → Compare → Investigate → Explain → Recommend**

A data analyst should not stop at *"sales decreased."* Continue asking:

**Where? → When? → Which segment/product/customer? → What variables are associated with it? → Is it an anomaly or recurring pattern? → What should the business investigate or do next?**



## Step 1 — Import libraries

In [37]:
import pandas as pd
import plotly.express as px
from IPython.display import display

## Step 2 — Load the dataset


In [38]:
df = pd.read_csv("listings.csv")


**Ask:** What does one row represent? How many records and columns do we have?

In [39]:
df.shape

(22708, 90)

In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22708 entries, 0 to 22707
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            22708 non-null  int64  
 1   listing_url                                   22708 non-null  str    
 2   scrape_id                                     22708 non-null  int64  
 3   last_scraped                                  22708 non-null  str    
 4   source                                        22708 non-null  str    
 5   name                                          22708 non-null  str    
 6   description                                   21823 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   22707 non-null  str    
 9   host_id                                       22708 non-null  int64  
 1

## Step 3 — Understand structure and data types

**Ask:** Are numeric fields numeric? Is the date actually stored as a date?


In [41]:
# Keep your original selection, but use names so column order cannot change it.
cols_to_drop = [
    'listing_url', 'neighborhood_overview', 'picture_url', 'host_profile_url',
    'host_since', 'host_about', 'host_response_time', 'host_response_rate',
    'host_acceptance_rate', 'host_thumbnail_url', 'host_picture_url',
    'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
    'neighbourhood', 'price_quote_price_per_night', 'price_quote_raw',
    'calendar_updated', 'instant_bookable'
]
df = df.drop(columns=cols_to_drop)
# Dropping a column is a scope choice; it does not mean that column is invalid.

**Convert** data to proper type

In [42]:
# Remove currency formatting; keep missing/unreadable prices as missing.
df['price'] = pd.to_numeric(
    df['price'].replace(r'[\$,]', '', regex=True), errors='coerce'
)

In [43]:
df['price'].dtypes

dtype('float64')

In [44]:
display(df.sample(5, random_state=42))
df.info()

,id,scrape_id,last_scraped,source,name,description,host_id,host_url,host_profile_id,host_name,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
20867,1241902600240413011,20260620164952,2026-06-21,city scrape,Wanderlust Gran Vía Apartments,Enjoy a luxury experience in this central acco...,265228558,https://www.airbnb.com/users/show/265228558,1.463338e+18,Hernando,...,4.17,4.50,4.92,4.17,NaN,2,2,0,0,0.67
22212,1187669688678088715,20260620164952,2026-06-22,city scrape,CituSpace Las Tablas-1 Bed with sofa bed & Par...,NaN,31772865,https://www.airbnb.com/users/show/31772865,1.465467e+18,Cituspace,...,4.40,4.40,4.60,4.60,Spain - National registration number<br />ESFC...,74,74,0,0,0.64
3108,1293407313235376262,20260620164952,2026-06-21,city scrape,Méndez Álvaro 1 Dormitorio,Discover a new way to experience Madrid at the...,662112418,https://www.airbnb.com/users/show/662112418,1.470662e+18,Plenora,...,4.91,4.77,4.67,4.54,Spain - National registration number<br />ESFC...,2,2,0,0,8.63
15447,891999910962966184,20260620164952,2026-07-02,previous scrape,Cómoda Habitación Madrid,Ideal room for 1 person.<br />Well lit with a ...,131659268,https://www.airbnb.com/users/show/131659268,1.468318e+18,Jorge,...,5.00,5.00,5.00,4.93,NaN,1,0,1,0,0.37
12049,45320031,20260620164952,2026-07-02,previous scrape,Habitacion para noches O dias.,NaN,365289537,https://www.airbnb.com/users/show/365289537,1.469735e+18,Fulmate,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN


<class 'pandas.DataFrame'>
RangeIndex: 22708 entries, 0 to 22707
Data columns (total 71 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            22708 non-null  int64  
 1   scrape_id                                     22708 non-null  int64  
 2   last_scraped                                  22708 non-null  str    
 3   source                                        22708 non-null  str    
 4   name                                          22708 non-null  str    
 5   description                                   21823 non-null  str    
 6   host_id                                       22708 non-null  int64  
 7   host_url                                      22708 non-null  str    
 8   host_profile_id                               22556 non-null  float64
 9   host_name                                     22556 non-null  str    
 1

In [45]:
# convert them to datetime format
# [last_scraped, price_quote_checkin_date, price_quote_checkout_date, calendar_last_scraped, first_review, last_review]
date_columns = [
    'last_scraped',
    'price_quote_checkin_date',
    'price_quote_checkout_date',
    'calendar_last_scraped',
    'first_review',
    'last_review'
]

df[date_columns] = df[date_columns].apply(pd.to_datetime, errors='coerce')

# ".apply()" is doing the function for each column consequently, working as for loop

In [46]:
df[date_columns].dtypes

last_scraped                 datetime64[us]
price_quote_checkin_date     datetime64[us]
price_quote_checkout_date    datetime64[us]
calendar_last_scraped        datetime64[us]
first_review                 datetime64[us]
last_review                  datetime64[us]
dtype: object

## Step 4 — Check missing values

**Ask:** Which columns have missing values, and what percentage is missing?


In [47]:
# Multiply by 100 to show percentages rather than proportions.
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
})
missing[missing['missing_count'] > 0].sort_values(
    'missing_percent', ascending=False
).round(2)

,missing_count,missing_percent
license,8056,35.48
host_location,7485,32.96
bedrooms,5719,25.18
bathrooms,5405,23.80
beds,4544,20.01
review_scores_value,3718,16.37
review_scores_location,3717,16.37
review_scores_cleanliness,3716,16.36
review_scores_communication,3716,16.36
review_scores_checkin,3716,16.36


## Step 5 — Check duplicates and uniqueness

**Ask:** Are there duplicated rows? 


In [48]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate listing IDs:', df['id'].duplicated().sum())

Duplicate rows: 0
Duplicate listing IDs: 0


## Step 6 — Clean obvious data-quality issues

For practice, we will:
- remove exact duplicate rows;
- fill missing categorical values with `"Unknown"`;

In [49]:
df = df.drop_duplicates()
category_columns = ['room_type', 'property_type', 'neighbourhood_cleansed']
# Assign back to df: inplace=True on df[[...]] does not update df reliably.
df[category_columns] = df[category_columns].fillna('Unknown')
# Leave missing numerical values as missing; zero would change their meaning.

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22708 entries, 0 to 22707
Data columns (total 71 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   id                                            22708 non-null  int64         
 1   scrape_id                                     22708 non-null  int64         
 2   last_scraped                                  22708 non-null  datetime64[us]
 3   source                                        22708 non-null  str           
 4   name                                          22708 non-null  str           
 5   description                                   21823 non-null  str           
 6   host_id                                       22708 non-null  int64         
 7   host_url                                      22708 non-null  str           
 8   host_profile_id                               22556 non-null  float64       


## Step 7 — Descriptive statistics

**Ask:** What are the typical values, ranges, and variability?


In [51]:
# IDs are numbers, but their averages and correlations are not meaningful.
numeric_cols = [
    'price', 'accommodates', 'bedrooms', 'bathrooms', 'beds',
    'review_scores_rating', 'number_of_reviews', 'availability_365',
    'minimum_nights'
]
df[numeric_cols].describe().round(2)

,price,accommodates,bedrooms,bathrooms,beds,review_scores_rating,number_of_reviews,availability_365,minimum_nights
count,19172.00,22708.00,16989.00,17303.00,18164.00,18993.00,22708.00,22708.00,22706.00
mean,180.06,3.25,1.63,1.31,2.09,4.64,62.35,191.82,7.73
std,555.62,1.95,1.07,0.75,2.23,0.48,108.83,127.88,23.25
min,3.36,1.00,0.00,0.50,1.00,1.00,0.00,0.00,1.00
25%,83.74,2.00,1.00,1.00,1.00,4.54,2.00,67.00,1.00
50%,125.00,3.00,1.00,1.00,2.00,4.75,17.00,209.00,2.00
75%,184.00,4.00,2.00,1.50,3.00,4.90,74.00,316.00,4.00
max,36883.00,16.00,50.00,38.00,167.00,5.00,1433.00,365.00,1125.00


In [52]:
df[category_columns].describe()

,room_type,property_type,neighbourhood_cleansed
count,22708,22708,22708
unique,4,59,128
top,Entire home/apt,Entire rental unit,Embajadores
freq,15899,14291,2453


## Step 8 — Explore categorical variables

**Ask:** Which room types, property types, and neighbourhoods have the most listings?
The charts show up to 12 categories by listing count, not by price.

In [53]:
categorical_cols = category_columns

In [54]:
df['property_type'].value_counts()

property_type
Entire rental unit                    14291
Private room in rental unit            4535
Private room in home                    565
Entire condo                            433
Room in hotel                           427
Entire loft                             384
Private room in hostel                  327
Entire serviced apartment               309
Entire home                             291
Private room in condo                   235
Private room in bed and breakfast       118
Room in aparthotel                       78
Private room in guesthouse               72
Private room in casa particular          67
Shared room in bed and breakfast         63
Private room in guest suite              49
Room in boutique hotel                   37
Private room                             35
Shared room in hotel                     31
Room in hostel                           30
Shared room in hostel                    27
Private room in serviced apartment       27
Entire guesthouse 

**Visualize** the categorical columns

In [55]:
for col in categorical_cols:
    value_counts = df[col].value_counts().head(12).reset_index()
    value_counts.columns = [col, 'Count']
    fig = px.bar(
        value_counts, x=col, y='Count', text_auto=True,
        title=f'{col} — up to 12 most common categories'
    )
    fig.show()

### Review of your work through step 8

**What works well:** You load and inspect the dataset, convert price and dates,
check missing values and duplicates, describe the data, and count categories.
This is a good EDA sequence. One row represents a listing; listing IDs are unique
in this file.

**Small fixes made:**
- Step 3 now drops columns by name and explicitly displays the sample. Your date
  conversion with `.apply(pd.to_datetime, errors='coerce')` is appropriate.
- Step 4 reports missing percentages: `.mean()` alone gives a fraction.
- Step 5 checks repeated listing IDs as well as repeated rows.
- Step 6 assigns `fillna()` back to the dataframe, fixing the saved chained-assignment
  warning. Missing numerical values are left missing.
- Step 7 focuses on useful measurements instead of IDs and summarizes categories.
- Step 8 keeps your frequency charts, with titles explaining the top-12 limit.

**Next:** Compare numerical variables, then prices across categories. A numeric ID
is not a measurement. Missing prices are not free stays, and missing ratings are
not zero-star ratings. The dropped columns remain in the original CSV if needed.

## Step 9 — Explore numerical distributions

**Ask:** What is a typical price? Are there unusually large values?

Use `df` for the full dataset and `priced` for listings with a recorded positive
price. Prices stay in the dataset's recorded units; removing a currency symbol
does not convert currencies or establish the quote basis.

In [56]:
print('All listings:', len(df))
print('Missing prices:', df['price'].isna().sum())
print('Zero or negative prices:', df['price'].le(0).sum())
priced = df[df['price'] > 0].copy()
print('Listings available for price analysis:', len(priced))
display(priced['price'].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

All listings: 22708
Missing prices: 3536
Zero or negative prices: 0
Listings available for price analysis: 19172


count    19172.00
mean       180.06
std        555.62
min          3.36
25%         83.74
50%        125.00
75%        184.00
95%        399.21
99%        919.99
max      36883.00
Name: price, dtype: float64

In [67]:
# Limit ONLY this histogram to the 99th percentile so typical prices are visible.
price_limit = priced['price'].quantile(0.99)
price_view = priced[priced['price'] <= price_limit]
print('Histogram upper limit:', round(price_limit, 2))
print('Listings above this limit (kept in all summaries):', len(priced) - len(price_view))
px.histogram(
    price_view, x='price', nbins=50, text_auto=True,
    title='Price distribution — up to the 99th percentile (chart only)'
).show()

for col in ['accommodates', 'bedrooms', 'review_scores_rating', 'number_of_reviews']:
    px.histogram(df, x=col, nbins=30, text_auto=True, title=f'Distribution of {col}').show()

Histogram upper limit: 919.99
Listings above this limit (kept in all summaries): 192


**Read the output:** A mean much higher than the median suggests a long right tail.
Check each column's `count`: pandas omits missing values from these summaries.

## Step 10 — Flag possible price outliers using IQR

IQR is the distance from the 25th to the 75th percentile. Values outside
`Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR` are worth inspecting, not automatically deleting.
Large properties or different room types may naturally have different prices.

In [58]:
q1 = priced['price'].quantile(0.25)
q3 = priced['price'].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
price_outliers = priced[
    (priced['price'] < lower_limit) | (priced['price'] > upper_limit)
]
print('IQR limits:', round(lower_limit, 2), 'to', round(upper_limit, 2))
print('Flagged listings:', len(price_outliers))
print('Percent of priced listings:', round(len(price_outliers) / len(priced) * 100, 2))
px.box(
    priced, y='price', points=False, log_y=True,
    title='All positive prices — logarithmic axis, individual points hidden'
).show()

IQR limits: -66.64 to 334.38
Flagged listings: 1369
Percent of priced listings: 7.14


## Step 11 — Numerical vs numerical relationships

**Ask:** Does price tend to increase with capacity, bedrooms, or bathrooms?
Is it associated with ratings, review counts, or available days?

Spearman correlation compares ranks and is useful for these skewed measurements.
Values near +1 mean the variables tend to rise together; values near -1 mean
they move in opposite directions. A value near 0 means a weak rank association.
Correlation does not show cause and effect. Missing values are excluded separately
for each pair, so check the pair counts too.

In [59]:
correlation = priced[numeric_cols].corr(method='spearman')
display(correlation['price'].drop('price').sort_values(ascending=False).round(3))

pair_counts = priced[numeric_cols].notna().astype(int)
pair_counts = pair_counts.T.dot(pair_counts)
display(pair_counts['price'].rename('listings_with_both_values').to_frame())

px.imshow(
    correlation, text_auto='.2f', color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, title='Spearman correlation — positive-price listings'
).show()

accommodates            0.642
beds                    0.549
bedrooms                0.462
bathrooms               0.297
number_of_reviews       0.121
review_scores_rating    0.082
availability_365        0.028
minimum_nights         -0.202
Name: price, dtype: float64

,listings_with_both_values
price,19172
accommodates,19172
bedrooms,14840
bathrooms,17029
beds,17888
review_scores_rating,16464
number_of_reviews,19172
availability_365,19172
minimum_nights,19171


In [60]:
# Sample only the scatter plots to keep the notebook responsive.
# All calculations above and below use the full eligible data.
scatter_data = priced.sample(n=min(2000, len(priced)), random_state=42)
print('Scatter plot sample:', len(scatter_data), 'listings')
for col in ['accommodates', 'bedrooms', 'review_scores_rating']:
    px.scatter(
        scatter_data, x=col, y='price', color='room_type', opacity=0.4,
        log_y=True, hover_data=['neighbourhood_cleansed'],
        title=f'Price vs {col} — sample, logarithmic price axis'
    ).show()

Scatter plot sample: 2000 listings


**Read the charts:** The logarithmic price axis includes large prices while making
typical listings easier to see. Equal vertical distances represent equal price
ratios, not equal price differences. Points with missing x-values are not drawn.
Available days are not the same as occupancy: unavailable dates may be booked or blocked.

## Step 12 — Numerical vs categorical: prices across groups

**Ask:** How do prices differ by room type, neighbourhood, and property type?

Use `groupby()` to calculate median and mean prices, and show how many prices are
available. Median is less affected by very expensive listings. The complete
tables include every category; the neighbourhood/property charts show the 12
categories with the most listings, ordered by median price.

In [61]:
# Include unpriced listings in group counts, but exclude invalid prices from summaries.
group_data = df.copy()
group_data['price'] = group_data['price'].where(group_data['price'] > 0)
price_summaries = {}

for col in category_columns:
    summary = group_data.groupby(col).agg(
        listings=('id', 'size'),
        priced_listings=('price', 'count'),
        median_price=('price', 'median'),
        mean_price=('price', 'mean')
    )
    summary['price_coverage_percent'] = summary['priced_listings'] / summary['listings'] * 100
    summary = summary.sort_values('median_price', ascending=False)
    price_summaries[col] = summary
    print(col)
    display(summary.round(2))

    common_groups = df[col].value_counts().head(12).index
    chart_summary = summary.loc[common_groups].sort_values('median_price').reset_index()
    px.bar(
        chart_summary, x='median_price', y=col, orientation='h', text_auto='.2f',
        hover_data=['listings', 'priced_listings', 'mean_price', 'price_coverage_percent'],
        title=f'Median price by {col} — up to 12 most common groups', height=500
    ).show()

room_type


,listings,priced_listings,median_price,mean_price,price_coverage_percent
room_type,,,,,
Entire home/apt,15899,13936,144.0,202.80,87.65
Hotel room,39,29,104.0,107.70,74.36
Private room,6611,5065,64.0,121.21,76.61
Shared room,159,142,25.0,62.00,89.31


property_type


,listings,priced_listings,median_price,mean_price,price_coverage_percent
property_type,,,,,
Entire villa,8,4,867.34,923.38,50.00
Private room in floor,4,2,700.40,700.40,50.00
Private room in cottage,1,1,359.50,359.50,100.00
Entire chalet,9,8,319.28,531.41,88.89
Religious building,2,2,264.00,264.00,100.00
Shared room in condo,2,2,260.15,260.15,100.00
Casa particular,6,4,207.74,378.12,66.67
Entire serviced apartment,309,269,173.00,360.75,87.06
Entire home,291,223,168.00,323.47,76.63


neighbourhood_cleansed


,listings,priced_listings,median_price,mean_price,price_coverage_percent
neighbourhood_cleansed,,,,,
Recoletos,307,264,193.68,257.48,85.99
Jerónimos,89,81,170.04,227.39,91.01
Goya,367,294,168.67,209.24,80.11
Castellana,179,147,164.63,223.16,82.12
Sol,1240,1125,163.88,237.01,90.73
...,...,...,...,...,...
Aguilas,49,40,52.85,80.91,81.63
Campamento,25,20,50.50,86.40,80.00
Apostol Santiago,34,30,48.50,82.48,88.24


In [62]:
# Box plots show spread and overlap, which median bars cannot show.
for col in category_columns:
    common_groups = df[col].value_counts().head(12).index
    box_data = priced[priced[col].isin(common_groups)]
    px.box(
        box_data, x=col, y='price', points=False, log_y=True,
        title=f'Price spread by {col} — common groups, logarithmic axis', height=500
    ).show()

**Interpret carefully:** A category with few recorded prices has a less stable median.
Different price coverage may also make comparisons unrepresentative. Property type
often already includes room type in its label, so these are related categories.

## Step 13 — Compare neighbourhoods within the same room type

**Ask:** Do neighbourhood differences remain when we compare similar room types?

The original template suggested a time series. This file is a listing snapshot
collected on a few dates, not repeated observations of each listing's price.
`last_review` is a latest-review date, not a booking or revenue history. A price
trend or seasonal sales conclusion would therefore be misleading here.

Instead, use a simple pivot table. Show the 12 most common neighbourhoods and
hide cells with fewer than 30 recorded prices. This is a practical display rule,
not a guarantee of statistical reliability.

In [63]:
common_neighbourhoods = df['neighbourhood_cleansed'].value_counts().head(12).index
comparison = priced[priced['neighbourhood_cleansed'].isin(common_neighbourhoods)]
median_by_room = comparison.pivot_table(
    index='neighbourhood_cleansed', columns='room_type', values='price', aggfunc='median'
)
count_by_room = comparison.pivot_table(
    index='neighbourhood_cleansed', columns='room_type', values='price',
    aggfunc='count', fill_value=0
)
display(count_by_room)
median_by_room = median_by_room.where(count_by_room >= 30)
display(median_by_room.round(2))
px.imshow(
    median_by_room, text_auto='.0f', color_continuous_scale='Blues',
    title='Median price by neighbourhood and room type — at least 30 prices per cell',
    aspect='auto', height=550
).show()

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Argüelles,144,0,103,3
Cortes,718,0,186,0
Cuatro Caminos,257,0,85,54
Embajadores,1630,8,387,23
Goya,257,0,37,0
Guindalera,289,0,43,12
Justicia,742,0,200,1
Palacio,1197,1,260,11
Palos de Moguer,227,0,75,3


room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Argüelles,160.33,NaN,43.16,NaN
Cortes,169.00,NaN,109.54,NaN
Cuatro Caminos,141.24,NaN,70.67,25.0
Embajadores,138.76,NaN,72.00,NaN
Goya,181.00,NaN,55.00,NaN
Guindalera,140.00,NaN,64.00,NaN
Justicia,169.50,NaN,84.42,NaN
Palacio,157.60,NaN,93.00,NaN
Palos de Moguer,139.00,NaN,69.00,NaN


Blank cells mean insufficient observations, not zero price. Comparing within room
type improves context, but property size, amenities, quote dates, and other
differences can still explain price gaps.

## Step 14 — Inspect extremes and missing-price patterns

**Ask:** Which listings need a closer look? Are missing prices concentrated in a group?
The most expensive listing is not necessarily the best performer: price is not profit,
revenue, or booking demand.

In [64]:
inspection_cols = [
    'id', 'name', 'price', 'room_type', 'property_type',
    'neighbourhood_cleansed', 'accommodates', 'bedrooms', 'bathrooms', 'beds'
]
print('10 highest recorded prices')
display(priced.nlargest(10, 'price')[inspection_cols])
print('10 lowest positive recorded prices')
display(priced.nsmallest(10, 'price')[inspection_cols])
print('Largest bedroom counts — inspect whether these are comparable listing units')
display(df.nlargest(5, 'bedrooms')[inspection_cols])

10 highest recorded prices


,id,name,price,room_type,property_type,neighbourhood_cleansed,accommodates,bedrooms,bathrooms,beds
20058,53056104,Madrid,36883.0,Entire home/apt,Entire rental unit,Universidad,6,3.0,1.5,NaN
162,34570478,Apartamento,22832.5,Entire home/apt,Entire loft,Sol,1,1.0,1.0,2.0
198,1496308350558748700,Plaza de la Villa doble con baño compartido,15000.0,Private room,Private room in home,Palacio,2,NaN,1.0,1.0
213,1482716640416035527,Centric 2 dormitorios,15000.0,Entire home/apt,Entire rental unit,Cortes,7,2.0,1.0,3.0
214,1496255451562926135,Centric 1 dormitorio,15000.0,Entire home/apt,Entire rental unit,Cortes,4,1.0,1.0,1.0
275,1498654933003051613,Buenavista apartamento,15000.0,Entire home/apt,Entire rental unit,Embajadores,4,1.0,1.0,1.0
6249,1496302830029735943,Espoz y Mina doble con baño compartido,15000.0,Private room,Private room in home,Cortes,2,NaN,1.0,1.0
6250,1496264019813624356,Núñez de Arce estudio,15000.0,Entire home/apt,Entire rental unit,Cortes,4,NaN,1.0,2.0
9465,1496323497168656654,Buenavista doble con baño compartido,15000.0,Private room,Private room in home,Embajadores,2,NaN,1.0,1.0
9466,1498653316318644494,Buenavista estudio,15000.0,Entire home/apt,Entire rental unit,Embajadores,3,1.0,1.0,1.0


10 lowest positive recorded prices


,id,name,price,room_type,property_type,neighbourhood_cleansed,accommodates,bedrooms,bathrooms,beds
17765,42080666,MADRID CENTRO DESING,3.36,Private room,Private room in rental unit,San Isidro,1,NaN,NaN,NaN
22308,1507595,Flat 3Rooms Ventas up to 6 easy to park,4.65,Entire home/apt,Entire rental unit,Ventas,6,3.0,NaN,NaN
5378,1681014813315383176,13D7 Habitación cama doble con aire acondicionado,5.03,Private room,Private room in rental unit,Cortes,1,1.0,1.0,1.0
8524,880795197907807332,Encantador Duplex pleno centro,6.89,Entire home/apt,Entire rental unit,Embajadores,3,1.0,NaN,NaN
6397,650669100988899030,Apartamento diseño industrial en zona financiera,7.02,Entire home/apt,Entire rental unit,Cuatro Caminos,4,1.0,NaN,NaN
8525,611601130764966973,Precioso duplex en Madrid Centro,7.40,Entire home/apt,Entire rental unit,Embajadores,3,1.0,NaN,NaN
8522,861371947458329526,Acogedor Duplex en pleno Centro,7.46,Entire home/apt,Entire rental unit,Embajadores,3,1.0,NaN,NaN
1382,1220239709744410594,Habitacion en Carabanchel,7.71,Private room,Private room in rental unit,Abrantes,1,NaN,NaN,NaN
4736,1369528898864816023,Habitación en piso compartido,8.01,Private room,Private room in rental unit,Castillejos,1,NaN,NaN,NaN
5083,978252148209220621,Piso de 3 habitaciones dobles cerca del Bernabéu.,8.02,Entire home/apt,Entire rental unit,Colina,5,3.0,NaN,NaN


Largest bedroom counts — inspect whether these are comparable listing units


,id,name,price,room_type,property_type,neighbourhood_cleansed,accommodates,bedrooms,bathrooms,beds
18023,1705305290816860090,Beddy Hostels,47.2,Private room,Room in hotel,Simancas,1,50.0,1.0,1.0
8739,51204846,Habitación privada de 6,170.0,Private room,Private room in hostel,Embajadores,6,25.0,1.0,167.0
18575,1296653058099912706,Hostal Patria Doble baño compartido,NaN,Private room,Room in hotel,Sol,2,18.0,NaN,NaN
18576,1295882263524337884,Hostal Patria Habitación doble,NaN,Private room,Room in hotel,Sol,2,18.0,NaN,NaN
18577,1296662804005576431,Hostal Patria Individual baño compartido.,NaN,Private room,Room in hotel,Sol,1,18.0,NaN,NaN


In [65]:
# Check whether missing price data is concentrated in one source.
missing_by_source = df.assign(price_missing=df['price'].isna()).groupby('source').agg(
    listings=('id', 'size'),
    missing_prices=('price_missing', 'sum'),
    missing_price_fraction=('price_missing', 'mean')
)
missing_by_source['missing_price_percent'] = missing_by_source.pop('missing_price_fraction') * 100
display(missing_by_source.round(2))

,listings,missing_prices,missing_price_percent
source,,,
city scrape,18548,285,1.54
previous scrape,4160,3251,78.15


## Step 15 — Summarize findings and next questions

The cell below calculates a short summary from the data, so it updates when rerun.
Use the tables and plots to explain both the differences and their limitations.

In [66]:
print(f"Missing prices: {df['price'].isna().mean():.1%} of all listings.")
print(f"Median price: {priced['price'].median():.2f}; mean price: {priced['price'].mean():.2f}.")
print('Median prices and available-price counts by room type:')
display(price_summaries['room_type'][['priced_listings', 'median_price']].round(2))
print('Numerical variables most strongly associated with price by absolute Spearman correlation:')
price_relationships = correlation['price'].drop('price').dropna()
display(price_relationships.reindex(price_relationships.abs().sort_values(ascending=False).index).head(3).round(3))

Missing prices: 15.6% of all listings.
Median price: 125.00; mean price: 180.06.
Median prices and available-price counts by room type:


,priced_listings,median_price
room_type,,
Entire home/apt,13936,144.0
Hotel room,29,104.0
Private room,5065,64.0
Shared room,142,25.0


Numerical variables most strongly associated with price by absolute Spearman correlation:


accommodates    0.642
beds            0.549
bedrooms        0.462
Name: price, dtype: float64

**Questions to answer from your results:**
1. How different are mean and median price? What does the distribution explain?
2. Which numerical variables have the strongest association with price?
3. Which room types have higher median prices, and how many recorded prices support each result?
4. Among the common neighbourhoods and property types, where are median prices higher or lower?
5. Do neighbourhood differences remain within the same room type?
6. Which unusual listings should be checked against the original records?
7. Which groups have low price coverage, and how might this affect the conclusions?

**Suggested follow-up:** Verify unusual listing sizes and price quote definitions,
then compare listings of similar capacity within room type and neighbourhood.
For seasonality, collect repeated dated prices or booking data. These comparisons
describe associations in observed listing prices; they do not measure profitability
or prove what causes a higher price.

### Findings from this file

- Of 22,708 listings, 19,172 have recorded positive prices; 15.6% have missing prices.
- Median price is 125.00, below the mean of 180.06. The long upper tail makes the median a useful typical-price summary.
- Median prices are 144.00 for entire homes/apartments, 104.00 for hotel rooms, 64.00 for private rooms, and 25.00 for shared rooms. Only 29 hotel-room listings have prices, so interpret that median cautiously.
- Price has its strongest positive Spearman association among the selected variables with capacity (0.642), followed by beds (0.549) and bedrooms (0.462). Rating has only a weak association (0.082). These are unadjusted associations.
- Missing prices are concentrated in previous-scrape listings (78.15% missing), compared with city-scrape listings (1.54%). The price comparisons therefore mainly describe listings with prices in the city scrape.

These observations describe the supplied CSV. Rerun the summary cells if the data changes.